In [1]:
from pyspark.sql import functions as F
from spark_session import get_spark

spark = get_spark("FastFood-P4-Graphes",
                  extra_packages=["graphframes:graphframes:0.8.3-spark3.5-s_2.12"])
sc = spark.sparkContext

# Rend le module Python graphframes (contenu dans le jar) importable
import glob, sys
for j in glob.glob("/root/.ivy2/jars/graphframes*.jar"):
    if j not in sys.path:
        sys.path.insert(0, j)
        sc.addPyFile(j)
from graphframes import GraphFrame
from graphframes.lib import AggregateMessages as AM
print("GraphFrames OK")

https://repos.spark-packages.org added as a remote repository with the name: repo-1
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ab79838b-69b9-487b-ba29-5c394bab012b;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
downloading https://repos.spark-packages.org/graphframes/graphframes/0.8.3-spark3.5-s_2.12/graphframes-0.8.3-spark3.5-s_2.12.jar ...
	[SUCCESSFUL ] graphframes#graphframes;0.8.3-spark3.5-s_2.12!graphframes.jar (168ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.16/slf4j-api-1.7.16.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;1.7.16!slf4j-api.jar (96ms)
:: resolution report :: resolve 3503ms :: artifacts dl 278ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.slf4j#slf4j-api;1.7.16 from ce

GraphFrames OK


In [2]:
silver = spark.read.parquet("s3a://lake/silver/orders_history/")
delivered = silver.filter(F.col("status") == "DELIVERED")

clients = (delivered.select(F.col("customer_id").alias("id"), "city").distinct()
           .withColumn("type", F.lit("client"))
           .withColumn("category", F.lit(None).cast("string"))
           .withColumn("name", F.lit(None).cast("string")))
restos = (delivered.select(F.col("restaurant_id").alias("id"), "city", "category", "name").distinct()
          .withColumn("type", F.lit("restaurant")))

vertices = clients.unionByName(restos).cache()
edges = (delivered.groupBy(F.col("customer_id").alias("src"), F.col("restaurant_id").alias("dst"))
         .agg(F.count("*").alias("nb_commandes"),
              F.round(F.sum("amount_mad"), 2).alias("montant_total"))
         .cache())

g = GraphFrame(vertices, edges)
print("Sommets :", g.vertices.count(), "| dont restaurants :", restos.count())
print("Arêtes  :", g.edges.count())

26/11/15 17:26:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Sommets : 5684 | dont restaurants : 200


[Stage 19:=============================>                            (2 + 2) / 4]

Arêtes  : 12551


In [3]:
print("Top 10 restaurants par nombre de clients distincts (inDegree)")
(g.inDegrees.join(restos, "id")
  .select("id", "name", "city", "category", "inDegree")
  .orderBy(F.desc("inDegree")).show(10, truncate=False))

print("Top 10 clients par nombre de restaurants différents (outDegree)")
g.outDegrees.orderBy(F.desc("outDegree")).show(10)

Top 10 restaurants par nombre de clients distincts (inDegree)


+-----+--------------------+----------+---------+--------+
|id   |name                |city      |category |inDegree|
+-----+--------------------+----------+---------+--------+
|R-199|Healthy Corniche 199|Casablanca|healthy  |732     |
|R-087|Sushi Palmier 87    |Casablanca|sushi    |365     |
|R-097|Asiatique Ocean 97  |Rabat     |asiatique|355     |
|R-151|Pizza Sahara 151    |Rabat     |pizza    |295     |
|R-061|Pizza Zellige 61    |Tanger    |pizza    |193     |
|R-110|Tacos Medina 110    |Marrakech |tacos    |174     |
|R-183|Burger Palmier 183  |Rabat     |burger   |167     |
|R-144|Pizza Majorelle 144 |Marrakech |pizza    |126     |
|R-034|Healthy Sahara 34   |Casablanca|healthy  |123     |
|R-099|Pizza Corniche 99   |Tanger    |pizza    |113     |
+-----+--------------------+----------+---------+--------+
only showing top 10 rows

Top 10 clients par nombre de restaurants différents (outDegree)
+-------+---------+
|     id|outDegree|
+-------+---------+
|C-01157|        7|
|C-0

In [4]:
pr = g.pageRank(resetProbability=0.15, maxIter=10)
top_pr = (pr.vertices.filter("type = 'restaurant'")
            .select("id", "name", "city", "category", F.round("pagerank", 3).alias("pagerank"))
            .orderBy(F.desc("pagerank")))
top_pr.show(10, truncate=False)

nb_cmd = edges.groupBy(F.col("dst").alias("id")).agg(F.sum("nb_commandes").alias("nb_livrees"))
(top_pr.limit(10).join(nb_cmd, "id").join(g.inDegrees, "id")
       .orderBy(F.desc("pagerank")).show(truncate=False))

+-----+--------------------+----------+---------+--------+
|id   |name                |city      |category |pagerank|
+-----+--------------------+----------+---------+--------+
|R-199|Healthy Corniche 199|Casablanca|healthy  |155.706 |
|R-097|Asiatique Ocean 97  |Rabat     |asiatique|75.1    |
|R-087|Sushi Palmier 87    |Casablanca|sushi    |69.55   |
|R-151|Pizza Sahara 151    |Rabat     |pizza    |59.646  |
|R-061|Pizza Zellige 61    |Tanger    |pizza    |40.806  |
|R-110|Tacos Medina 110    |Marrakech |tacos    |35.451  |
|R-183|Burger Palmier 183  |Rabat     |burger   |34.415  |
|R-034|Healthy Sahara 34   |Casablanca|healthy  |25.994  |
|R-144|Pizza Majorelle 144 |Marrakech |pizza    |24.301  |
|R-099|Pizza Corniche 99   |Tanger    |pizza    |24.227  |
+-----+--------------------+----------+---------+--------+
only showing top 10 rows

+-----+--------------------+----------+---------+--------+----------+--------+
|id   |name                |city      |category |pagerank|nb_livrees|

In [5]:
sc.setCheckpointDir("s3a://lake/checkpoints/graphframes")
cc = g.connectedComponents()
tailles = cc.groupBy("component").agg(F.count("*").alias("taille"),
                                       F.collect_set("city").alias("villes"))
print("Nombre de composantes :", tailles.count())
tailles.orderBy(F.desc("taille")).show(truncate=False)

Nombre de composantes : 4


+---------+------+------------+
|component|taille|villes      |
+---------+------+------------+
|1        |2289  |[Casablanca]|
|2        |1415  |[Rabat]     |
|0        |1135  |[Marrakech] |
|13       |845   |[Tanger]    |
+---------+------+------------+



In [6]:
motifs = (g.find("(c)-[e1]->(r1); (c)-[e2]->(r2)")
           .filter("r1.category = r2.category AND r1.id < r2.id"))
print("Couples (client, 2 restaurants de même catégorie) :", motifs.count())
motifs.select("c.id", "r1.id", "r1.name", "r2.id", "r2.name", "r1.category").show(10, truncate=False)

pairs = motifs.select(F.col("r1.id").alias("r1"), F.col("r2.id").alias("r2"))
co = (pairs.unionByName(pairs.select(F.col("r2").alias("r1"), F.col("r1").alias("r2")))
           .groupBy("r1", "r2").agg(F.count("*").alias("score")))

client = motifs.select("c.id").first()[0]
deja = edges.filter(F.col("src") == client).select(F.col("dst").alias("r1"))
reco = (co.join(deja, "r1")
          .groupBy("r2").agg(F.sum("score").alias("score"))
          .join(deja.withColumnRenamed("r1", "r2"), "r2", "left_anti")
          .join(restos.select(F.col("id").alias("r2"), "name", "category"), "r2")
          .orderBy(F.desc("score")))
print(f"Client {client} a déjà commandé chez :", [r[0] for r in deja.collect()])
print("Recommandations :")
reco.show(5, truncate=False)

Couples (client, 2 restaurants de même catégorie) : 1474
+-------+-----+--------------------+-----+--------------------+--------+
|id     |id   |name                |id   |name                |category|
+-------+-----+--------------------+-----+--------------------+--------+
|C-04151|R-146|Healthy Corniche 146|R-165|Healthy Atlas 165   |healthy |
|C-00586|R-013|Burger Andalous 13  |R-192|Burger Souk 192     |burger  |
|C-00998|R-074|Marocain Zellige 74 |R-197|Marocain Zellige 197|marocain|
|C-04596|R-004|Healthy Majorelle 4 |R-137|Healthy Hassan 137  |healthy |
|C-03833|R-054|Burger Atlas 54     |R-182|Burger Kasbah 182   |burger  |
|C-02943|R-017|Burger Souk 17      |R-054|Burger Atlas 54     |burger  |
|C-00066|R-183|Burger Palmier 183  |R-192|Burger Souk 192     |burger  |
|C-02992|R-017|Burger Souk 17      |R-158|Burger Atlas 158    |burger  |
|C-05035|R-129|Healthy Ocean 129   |R-146|Healthy Corniche 146|healthy |
|C-00423|R-048|Pizza Corniche 48   |R-200|Pizza Medina 200    |pizz

In [7]:
ca_msg = g.aggregateMessages(F.sum(AM.msg).alias("ca_recu"),
                             sendToDst=AM.edge["montant_total"])
ca_msg = ca_msg.join(restos.select("id", "name"), "id")
ca_msg.orderBy(F.desc("ca_recu")).show(10, truncate=False)

ca_groupby = edges.groupBy(F.col("dst").alias("id")).agg(F.sum("montant_total").alias("ca_groupby"))
diff = (ca_msg.join(ca_groupby, "id")
              .filter(F.abs(F.col("ca_recu") - F.col("ca_groupby")) > 0.01).count())
print("Restaurants avec un écart aggregateMessages / groupBy :", diff)

+-----+------------------+-----------------------+
|id   |ca_recu           |name                   |
+-----+------------------+-----------------------+
|R-199|211248.69999999998|Healthy Corniche 199   |
|R-087|130939.26000000001|Sushi Palmier 87       |
|R-097|110484.73000000001|Asiatique Ocean 97     |
|R-151|80220.93          |Pizza Sahara 151       |
|R-061|52893.91          |Pizza Zellige 61       |
|R-183|37517.13          |Burger Palmier 183     |
|R-144|36595.89          |Pizza Majorelle 144    |
|R-110|34304.5           |Tacos Medina 110       |
|R-196|33677.22          |Asiatique Majorelle 196|
|R-034|32326.670000000002|Healthy Sahara 34      |
+-----+------------------+-----------------------+
only showing top 10 rows

Restaurants avec un écart aggregateMessages / groupBy : 0


In [8]:
spark.stop()

## Partie 4 – Analyse

### Le graphe
5 684 sommets (5 484 clients, 200 restaurants) et 12 551 arêtes orientées client → restaurant, pondérées par le nombre de commandes et le montant total. Le graphe est **biparti** : aucune arête client–client ni restaurant–restaurant.

### Degrés
- **inDegree** (clients distincts par restaurant) : R-199 Healthy Corniche (732) domine largement, devant R-087 (365) et R-097 (355).
- **outDegree** (restaurants différents par client) : 7 au maximum. Les clients sont fidèles à quelques adresses de leur ville.

### PageRank et comparaison avec le top 5 de la Partie 2
| Rang | Partie 2 (commandes livrées) | PageRank |
|---|---|---|
| 1 | R-199 (907) | R-199 (155,7) |
| 2 | R-097 (416) | R-097 (75,1) |
| 3 | R-087 (412) | R-087 (69,6) |
| 4 | R-151 (341) | R-151 (59,6) |
| 5 | R-061 (223) | R-061 (40,8) |

Les deux classements se ressemblent, mais ils ne mesurent pas la même chose :
- La Partie 2 compte le **volume** de commandes : un client qui commande 10 fois pèse 10 fois.
- PageRank ignore le poids des arêtes : un client compte **une seule fois** par restaurant. Il transmet son score **divisé par son nombre de restaurants** (outDegree). Le score d'un restaurant est donc élevé s'il a beaucoup de clients **distincts** et **fidèles** (qui fréquentent peu d'autres restaurants).
- Exemple : R-087 a plus de clients distincts que R-097 (365 contre 355), mais un PageRank plus faible (69,6 contre 75,1). Les clients de R-097 sont plus exclusifs : leur vote est moins dilué.

PageRank mesure donc l'**attachement de la clientèle** plutôt que le chiffre d'affaires.

### Composantes connexes
4 composantes, une par ville. Un client ne commande que dans sa ville, donc aucune arête ne relie deux villes. Pour l'entreprise, cela signifie 4 marchés indépendants, et la réaffectation des livreurs se raisonne ville par ville. `connectedComponents()` exige un répertoire de checkpoint (`sc.setCheckpointDir`) : l'algorithme est itératif, et le checkpoint tronque le lineage, qui grandirait à chaque itération.

### Motifs et recommandation
Le motif `(c)-[e1]->(r1); (c)-[e2]->(r2)` filtré sur `r1.category = r2.category` donne 1 474 couples de restaurants concurrents partageant un client. On en déduit une **co-fréquentation** : « les clients de r1 vont aussi chez r2 ». Pour le client C-04151 (déjà client de R-146, R-165 et R-197), on recommande les restaurants *healthy* les plus co-fréquentés avec ceux qu'il connaît, et qu'il n'a pas encore essayés : R-129, R-004 et R-031 (score 6).

### aggregateMessages
Chaque arête envoie son `montant_total` vers sa destination (`sendToDst`), et chaque restaurant somme les messages reçus (`F.sum(AM.msg)`). Le résultat est identique au `groupBy("dst").sum()`, avec **0 écart**. Les décimales parasites (211248.69999999998) viennent de l'arithmétique flottante ; on les supprime avec `F.round(..., 2)`.

### GraphX ou GraphFrames
| | GraphX | GraphFrames |
|---|---|---|
| API | RDD, `Graph[VD, ED]` typé | DataFrames `vertices(id, …)` / `edges(src, dst, …)` |
| Langage | Scala/Java uniquement | Scala, Java, **Python** |
| Optimisation | aucune (lambdas opaques) | **Catalyst** et Tungsten sur les requêtes |
| Requêtes | algorithmes et `Pregel` | algorithmes, **recherche de motifs** (`find`), filtres SQL |

**GraphFrames est utilisé en PySpark** car GraphX n'a pas d'API Python. GraphFrames encapsule GraphX : PageRank et les composantes connexes appellent GraphX sous le capot, mais l'utilisateur manipule des DataFrames.

### Le modèle Pregel
Pregel est un modèle de calcul **« penser comme un sommet »**, organisé en **super-étapes** synchronisées :
1. chaque sommet actif **envoie des messages** le long de ses arêtes ;
2. chaque sommet **agrège** les messages reçus (somme, min…) ;
3. chaque sommet **met à jour son état** à partir de cet agrégat ;
4. on recommence jusqu'à convergence ou jusqu'à `maxIter`.

- **PageRank** : à chaque itération, un sommet envoie `rang / outDegree` à ses voisins, et chacun calcule `rang = 0.15 + 0.85 × Σ messages`. Ici, 10 itérations.
- **aggregateMessages** est **une seule super-étape** Pregel : envoi le long des arêtes, puis agrégation à la destination.
- **connectedComponents** propage le plus petit identifiant de composante jusqu'à stabilité.